In [ ]:
# Cell 1 — Imports
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.feature_selection import RFE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
# Cell 2 — Initialise Logistic Regression with explicit params
# FIX: max_iter=1000 eliminates the ConvergenceWarning (default 100 is too low)
# FIX: class_weight='balanced' accounts for churn imbalance (~73% No, ~27% Yes)
# FIX: random_state=42 ensures reproducibility
# C=1.0 is standard L2 regularisation strength (inverse — lower = stronger)
model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    solver='lbfgs'
)

In [ ]:
# Cell 3 — Recursive Feature Elimination (RFE)
# FIX: n_features_to_select=10 set explicitly instead of relying on the default
#      (default is None = half of features, which was ~11 — now it's intentional)
# step=1 removes one feature per iteration (thorough)
rfe = RFE(estimator=model, n_features_to_select=10, step=1)
rfe.fit(X_train, y_train)

In [ ]:
# Cell 4 — Extract RFE-selected feature names
s = []
for code, classy in enumerate(rfe.support_):
    if classy:
        s.append(X_train.columns[code])

print('Selected features:', s)

In [ ]:
# Cell 5 — Align X_train & X_test to RFE-selected features, then fit
# FIX: X_test must be filtered to the same columns as X_train before predict()
#      Without this, model.predict(X_test) throws a shape mismatch error
X_train = X_train[s]
X_test  = X_test[s]   # FIX: critical alignment step

model.fit(X_train, y_train)

In [ ]:
# Cell 6 — Predict
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # probability scores for ROC-AUC

In [ ]:
# Cell 7 — Classification Report
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Cell 8 — Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — ROC-AUC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-AUC Curve — Logistic Regression')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()
print(f'AUC Score: {roc_auc:.4f}')

In [ ]:
# Cell 10 — Feature Coefficients (model interpretability)
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='coolwarm')
plt.title('Logistic Regression Coefficients')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()